In [ ]:
# UWB LOS/NLOS Classification - Complete Pipeline
# Optimized for NVIDIA RTX 5070 Ti
# CSC3105 Data Analytics and AI Project

"""
Setup Instructions:
1. Create conda environment:
   conda create -n uwb python=3.10
   conda activate uwb
   
2. Install PyTorch with CUDA:
   conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
   
3. Install other packages:
   pip install pandas numpy matplotlib seaborn scikit-learn xgboost jupyterlab tqdm

4. Verify GPU:
   python -c "import torch; print(f'CUDA Available: {torch.cuda.is_available()}')"
   python -c "import torch; print(f'GPU: {torch.cuda.get_device_name(0)}')"
"""


In [ ]:
# SECTION 1: IMPORTS AND SETUP


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import xgboost as xgb

# Deep Learning imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import math

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Check GPU
print("=" * 70)
print("SYSTEM CONFIGURATION")
print("=" * 70)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not available! Will use CPU (slow)")
print("=" * 70)

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
# SECTION 2: PROJECT STRUCTURE AND PATHS


In [ ]:
# Base paths
BASE_DIR = Path("Project")
DATASET_DIR = BASE_DIR / "UWB-LOS-NLOS-Data-Set" / "dataset"
RESULTS_DIR = BASE_DIR / "results"
MODELS_DIR = RESULTS_DIR / "models"
FIGURES_DIR = RESULTS_DIR / "figures"

# Create directories if they don't exist
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"\nProject Directory: {BASE_DIR.absolute()}")
print(f"Dataset Directory: {DATASET_DIR.absolute()}")
print(f"Results Directory: {RESULTS_DIR.absolute()}")

# Verify dataset exists
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
else:
    print(f"\n✓ Dataset directory found!")
    print(f"  Contents: {list(DATASET_DIR.glob('*'))[:5]}...")  # Show first 5 files


In [ ]:
# SECTION 3: DATA LOADING


In [ ]:
print("\n" + "=" * 70)
print("LOADING DATASET")
print("=" * 70)

def load_uwb_dataset(dataset_dir):
    """
    Load all UWB LOS/NLOS data from the dataset directory
    
    Args:
        dataset_dir: Path to dataset folder
    
    Returns:
        DataFrame with all samples
    """
    all_files = list(Path(dataset_dir).glob("*.csv"))
    
    if len(all_files) == 0:
        raise FileNotFoundError(f"No CSV files found in {dataset_dir}")
    
    print(f"Found {len(all_files)} CSV files")
    
    # Load all CSV files
    dataframes = []
    for file in tqdm(all_files, desc="Loading files"):
        try:
            df_temp = pd.read_csv(file)
            # Add environment name from filename
            env_name = file.stem  # filename without extension
            df_temp['environment'] = env_name
            dataframes.append(df_temp)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    # Combine all dataframes
    df = pd.concat(dataframes, ignore_index=True)
    
    print(f"\n✓ Loaded {len(df):,} total samples")
    print(f"  Columns: {list(df.columns)}")
    
    return df

# Load the dataset
df_raw = load_uwb_dataset(DATASET_DIR)

# Display basic info
print(f"\nDataset Shape: {df_raw.shape}")
print(f"\nFirst few rows:")
print(df_raw.head())

print(f"\nData types:")
print(df_raw.dtypes)


In [ ]:
# SECTION 4: EXPLORATORY DATA ANALYSIS (EDA)


In [ ]:
print("\n" + "=" * 70)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# Basic statistics
print("\n1. Dataset Overview:")
print(f"   Total Samples: {len(df_raw):,}")
print(f"   Features: {df_raw.shape[1]}")
print(f"   Missing Values: {df_raw.isnull().sum().sum()}")

# Class distribution
print("\n2. Class Distribution:")
if 'Class' in df_raw.columns:
    class_counts = df_raw['Class'].value_counts()
    print(f"   LOS (0): {class_counts.get(0, 0):,} samples")
    print(f"   NLOS (1): {class_counts.get(1, 0):,} samples")
    print(f"   Balance: {class_counts.get(0, 0) / class_counts.get(1, 1):.2f}")
else:
    print("   WARNING: 'Class' column not found!")

# Environment distribution
print("\n3. Environment Distribution:")
if 'environment' in df_raw.columns:
    env_counts = df_raw['environment'].value_counts()
    for env, count in env_counts.items():
        print(f"   {env}: {count:,} samples")

# Check for CIR column
print("\n4. CIR Data:")
cir_column = None
for col in df_raw.columns:
    if 'CIR' in col.upper() or 'cir' in col:
        cir_column = col
        break

if cir_column:
    print(f"   ✓ Found CIR column: '{cir_column}'")
    # Check CIR format
    sample_cir = df_raw[cir_column].iloc[0]
    print(f"   CIR type: {type(sample_cir)}")
    if isinstance(sample_cir, str):
        # Parse string to array
        cir_array = np.fromstring(sample_cir.strip('[]'), sep=',')
        print(f"   CIR length: {len(cir_array)} samples")
    elif isinstance(sample_cir, (list, np.ndarray)):
        print(f"   CIR length: {len(sample_cir)} samples")
else:
    print("   WARNING: CIR column not found!")
    print(f"   Available columns: {list(df_raw.columns)}")

# Statistical summary
print("\n5. Statistical Summary:")
print(df_raw.describe())


In [ ]:
# SECTION 5: DATA VISUALIZATION


In [ ]:
print("\n" + "=" * 70)
print("GENERATING VISUALIZATIONS")
print("=" * 70)

# Create visualization grid
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('UWB Dataset Exploratory Data Analysis', fontsize=16, y=1.02)

# 1. Class distribution
if 'Class' in df_raw.columns:
    class_counts = df_raw['Class'].value_counts()
    axes[0, 0].bar(['LOS', 'NLOS'], class_counts.values, color=['green', 'red'], alpha=0.7)
    axes[0, 0].set_title('LOS vs NLOS Distribution')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].grid(alpha=0.3)
    for i, v in enumerate(class_counts.values):
        axes[0, 0].text(i, v + 100, f'{v:,}', ha='center', va='bottom')

# 2. Environment distribution
if 'environment' in df_raw.columns:
    env_counts = df_raw['environment'].value_counts()
    axes[0, 1].barh(range(len(env_counts)), env_counts.values)
    axes[0, 1].set_yticks(range(len(env_counts)))
    axes[0, 1].set_yticklabels(env_counts.index, fontsize=8)
    axes[0, 1].set_title('Samples per Environment')
    axes[0, 1].set_xlabel('Count')
    axes[0, 1].grid(alpha=0.3, axis='x')

# 3. Sample CIR signals (LOS vs NLOS)
if cir_column and 'Class' in df_raw.columns:
    # Get sample CIRs
    los_sample = df_raw[df_raw['Class'] == 0][cir_column].iloc[0]
    nlos_sample = df_raw[df_raw['Class'] == 1][cir_column].iloc[0]
    
    # Parse if string
    if isinstance(los_sample, str):
        los_cir = np.fromstring(los_sample.strip('[]'), sep=',')
        nlos_cir = np.fromstring(nlos_sample.strip('[]'), sep=',')
    else:
        los_cir = np.array(los_sample)
        nlos_cir = np.array(nlos_sample)
    
    # Plot LOS
    axes[0, 2].plot(los_cir, label='LOS', color='green', linewidth=1)
    axes[0, 2].set_title('LOS CIR Example')
    axes[0, 2].set_xlabel('Time (ns)')
    axes[0, 2].set_ylabel('Amplitude')
    axes[0, 2].grid(alpha=0.3)
    axes[0, 2].legend()
    
    # Plot NLOS
    axes[1, 0].plot(nlos_cir, label='NLOS', color='red', linewidth=1)
    axes[1, 0].set_title('NLOS CIR Example')
    axes[1, 0].set_xlabel('Time (ns)')
    axes[1, 0].set_ylabel('Amplitude')
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].legend()
    
    # Overlay comparison
    axes[1, 1].plot(los_cir[:300], label='LOS', color='green', alpha=0.7, linewidth=1.5)
    axes[1, 1].plot(nlos_cir[:300], label='NLOS', color='red', alpha=0.7, linewidth=1.5)
    axes[1, 1].set_title('LOS vs NLOS Comparison (First 300 samples)')
    axes[1, 1].set_xlabel('Time (ns)')
    axes[1, 1].set_ylabel('Amplitude')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

# 4. Missing values heatmap
missing_data = df_raw.isnull().sum()
if missing_data.sum() > 0:
    axes[1, 2].barh(range(len(missing_data[missing_data > 0])), 
                     missing_data[missing_data > 0].values)
    axes[1, 2].set_yticks(range(len(missing_data[missing_data > 0])))
    axes[1, 2].set_yticklabels(missing_data[missing_data > 0].index)
    axes[1, 2].set_title('Missing Values')
    axes[1, 2].set_xlabel('Count')
else:
    axes[1, 2].text(0.5, 0.5, 'No Missing Values ✓', 
                     ha='center', va='center', fontsize=14)
    axes[1, 2].set_title('Missing Values Check')
axes[1, 2].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_EDA_overview.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '01_EDA_overview.png'}")
plt.show()


In [ ]:
# SECTION 6: FEATURE ENGINEERING FROM CIR


In [ ]:
print("\n" + "=" * 70)
print("FEATURE ENGINEERING FROM CIR")
print("=" * 70)

def extract_cir_features(cir_data):
    """
    Extract statistical features from CIR array
    
    Args:
        cir_data: CIR array (can be string, list, or numpy array)
    
    Returns:
        Dictionary of features
    """
    # Parse CIR if it's a string
    if isinstance(cir_data, str):
        cir = np.fromstring(cir_data.strip('[]'), sep=',')
    else:
        cir = np.array(cir_data)
    
    features = {}
    
    # Basic statistics
    features['cir_mean'] = np.mean(cir)
    features['cir_std'] = np.std(cir)
    features['cir_median'] = np.median(cir)
    features['cir_max'] = np.max(cir)
    features['cir_min'] = np.min(cir)
    features['cir_range'] = features['cir_max'] - features['cir_min']
    
    # Percentiles
    features['cir_q25'] = np.percentile(cir, 25)
    features['cir_q75'] = np.percentile(cir, 75)
    features['cir_iqr'] = features['cir_q75'] - features['cir_q25']
    
    # Energy features
    features['cir_energy'] = np.sum(cir ** 2)
    features['cir_rms'] = np.sqrt(np.mean(cir ** 2))
    
    # Peak features
    features['cir_peak_value'] = np.max(cir)
    features['cir_peak_index'] = np.argmax(cir)
    features['cir_peak_to_avg'] = features['cir_peak_value'] / (features['cir_mean'] + 1e-10)
    
    # Shape features
    from scipy.stats import skew, kurtosis
    features['cir_skewness'] = skew(cir)
    features['cir_kurtosis'] = kurtosis(cir)
    
    # Count significant peaks
    threshold = 0.5 * features['cir_peak_value']
    features['cir_num_peaks'] = np.sum(cir > threshold)
    
    # Rising edge features
    diff = np.diff(cir)
    features['cir_mean_rising_slope'] = np.mean(diff[diff > 0]) if np.any(diff > 0) else 0
    features['cir_max_rising_slope'] = np.max(diff) if len(diff) > 0 else 0
    
    # Zero crossings
    features['cir_zero_crossings'] = np.sum(np.diff(np.sign(cir - np.mean(cir))) != 0)
    
    # Centroid and spread
    indices = np.arange(len(cir))
    total_energy = np.sum(cir) + 1e-10
    features['cir_centroid'] = np.sum(indices * cir) / total_energy
    features['cir_spread'] = np.sqrt(np.sum((indices - features['cir_centroid']) ** 2 * cir) / total_energy)
    
    return features

# Extract features for all samples
print("Extracting CIR features for all samples...")
cir_features_list = []

for idx in tqdm(range(len(df_raw)), desc="Processing CIR"):
    cir_data = df_raw[cir_column].iloc[idx]
    features = extract_cir_features(cir_data)
    cir_features_list.append(features)

# Convert to DataFrame
df_cir_features = pd.DataFrame(cir_features_list)

print(f"\n✓ Extracted {len(df_cir_features.columns)} features from CIR")
print(f"  Features: {list(df_cir_features.columns)[:10]}...")

# Combine with original data (excluding CIR column)
original_features = [col for col in df_raw.columns if col != cir_column]
df_combined = pd.concat([
    df_raw[original_features].reset_index(drop=True),
    df_cir_features
], axis=1)

print(f"\n✓ Combined dataset shape: {df_combined.shape}")
print(f"  Total features: {df_combined.shape[1]}")

# Save feature descriptions
feature_descriptions = pd.DataFrame({
    'Feature': df_cir_features.columns,
    'Description': [
        'Mean amplitude of CIR',
        'Standard deviation of CIR',
        'Median amplitude',
        'Maximum amplitude',
        'Minimum amplitude',
        'Range (max - min)',
        '25th percentile',
        '75th percentile',
        'Interquartile range',
        'Total energy (sum of squares)',
        'Root mean square',
        'Peak value',
        'Peak index (time)',
        'Peak-to-average ratio',
        'Skewness (asymmetry)',
        'Kurtosis (tailedness)',
        'Number of significant peaks',
        'Mean rising slope',
        'Maximum rising slope',
        'Zero crossings',
        'Centroid (weighted mean time)',
        'Spread (variance in time)'
    ]
})

feature_descriptions.to_csv(RESULTS_DIR / 'cir_feature_descriptions.csv', index=False)
print(f"✓ Saved feature descriptions")


In [ ]:
# SECTION 7: DATA PREPROCESSING


In [ ]:
print("\n" + "=" * 70)
print("DATA PREPROCESSING")
print("=" * 70)

# Separate features and target
feature_cols = [col for col in df_combined.columns 
                if col not in ['Class', 'environment', cir_column]]

X = df_combined[feature_cols].copy()
y = df_combined['Class'].copy()

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Handle missing values
print("\n1. Handling missing values...")
missing_before = X.isnull().sum().sum()
if missing_before > 0:
    print(f"   Found {missing_before} missing values")
    X = X.fillna(X.median())
    print(f"   ✓ Filled with median values")
else:
    print(f"   ✓ No missing values found")

# Check for infinite values
inf_count = np.isinf(X.values).sum()
if inf_count > 0:
    print(f"   Found {inf_count} infinite values")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    print(f"   ✓ Replaced infinite values")

# Split data
print("\n2. Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"   Training set: {X_train.shape[0]:,} samples")
print(f"   Test set: {X_test.shape[0]:,} samples")
print(f"   Train LOS/NLOS: {np.bincount(y_train)}")
print(f"   Test LOS/NLOS: {np.bincount(y_test)}")

# Feature scaling
print("\n3. Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for interpretability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"   ✓ Scaled using StandardScaler")
print(f"   Mean: {X_train_scaled.mean().mean():.6f}")
print(f"   Std: {X_train_scaled.std().mean():.6f}")

# Save preprocessed data
np.save(RESULTS_DIR / 'X_train_scaled.npy', X_train_scaled.values)
np.save(RESULTS_DIR / 'X_test_scaled.npy', X_test_scaled.values)
np.save(RESULTS_DIR / 'y_train.npy', y_train.values)
np.save(RESULTS_DIR / 'y_test.npy', y_test.values)
print(f"\n✓ Saved preprocessed data to {RESULTS_DIR}")


In [ ]:
# SECTION 8: FEATURE IMPORTANCE ANALYSIS


In [ ]:
print("\n" + "=" * 70)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

# Train Random Forest for feature importance
print("Training Random Forest for feature importance...")
rf_importance = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_importance.fit(X_train_scaled, y_train)

# Get feature importances
importances = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': rf_importance.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 20 Most Important Features:")
print(importances.head(20).to_string(index=False))

# Visualize
plt.figure(figsize=(12, 8))
top_n = 20
plt.barh(range(top_n), importances.head(top_n)['importance'].values[::-1])
plt.yticks(range(top_n), importances.head(top_n)['feature'].values[::-1])
plt.xlabel('Importance')
plt.title(f'Top {top_n} Most Important Features')
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_feature_importance.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '02_feature_importance.png'}")
plt.show()

# Save importance scores
importances.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)


In [ ]:
# SECTION 9: BASELINE MODEL - XGBOOST


In [ ]:
print("\n" + "=" * 70)
print("TRAINING XGBOOST CLASSIFIER (BASELINE)")
print("=" * 70)

# Initialize XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    eval_metric='logloss',
    tree_method='hist',  # Faster on GPU
    device='cuda'  # Use GPU
)

# Train
print("Training XGBoost...")
xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)

# Predictions
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("\n" + "=" * 70)
print("XGBOOST RESULTS")
print("=" * 70)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['LOS', 'NLOS']))

print("\nConfusion Matrix:")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)

print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_xgb):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_xgb):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_xgb):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

# Save model
xgb_model.save_model(str(MODELS_DIR / 'xgboost_model.json'))
print(f"\n✓ Saved model to {MODELS_DIR / 'xgboost_model.json'}")

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['LOS', 'NLOS'], yticklabels=['LOS', 'NLOS'])
axes[0].set_title('XGBoost Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_xgb)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'XGBoost (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - XGBoost')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_xgboost_results.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '03_xgboost_results.png'}")
plt.show()


In [ ]:
# SECTION 10: TRANSFORMER MODEL IMPLEMENTATION


In [ ]:
print("\n" + "=" * 70)
print("TRANSFORMER MODEL IMPLEMENTATION")
print("=" * 70)

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1016):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# CIR Transformer Model
class CIRTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=3, 
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Linear(1, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x):
        # x: [batch, 1016]
        x = x.unsqueeze(-1)  # [batch, 1016, 1]
        x = self.embedding(x)  # [batch, 1016, d_model]
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.transpose(1, 2)  # [batch, d_model, 1016]
        x = self.global_pool(x).squeeze(-1)  # [batch, d_model]
        output = self.classifier(x)  # [batch, 1]
        return output

# Create model
print("Creating Transformer model...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CIRTransformer(d_model=128, nhead=4, num_layers=3).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model created on {device}")
print(f"  Total parameters: {total_params:,}")
print(f"  Model size: {total_params * 4 / 1e6:.2f} MB (FP32)")


In [ ]:
# SECTION 11: PREPARE DATA FOR TRANSFORMER


In [ ]:
print("\n" + "=" * 70)
print("PREPARING DATA FOR TRANSFORMER")
print("=" * 70)

# We need raw CIR data for transformer
print("Extracting raw CIR sequences...")
X_train_cir_list = []
X_test_cir_list = []

# Get indices
train_indices = X_train.index
test_indices = X_test.index

for idx in tqdm(train_indices, desc="Train CIR"):
    cir_data = df_raw.loc[idx, cir_column]
    if isinstance(cir_data, str):
        cir = np.fromstring(cir_data.strip('[]'), sep=',')
    else:
        cir = np.array(cir_data)
    X_train_cir_list.append(cir)

for idx in tqdm(test_indices, desc="Test CIR"):
    cir_data = df_raw.loc[idx, cir_column]
    if isinstance(cir_data, str):
        cir = np.fromstring(cir_data.strip('[]'), sep=',')
    else:
        cir = np.array(cir_data)
    X_test_cir_list.append(cir)

# Convert to numpy arrays
X_train_cir = np.array(X_train_cir_list)
X_test_cir = np.array(X_test_cir_list)

print(f"\nTrain CIR shape: {X_train_cir.shape}")
print(f"Test CIR shape: {X_test_cir.shape}")

# Normalize CIR
print("\nNormalizing CIR sequences...")
X_train_cir_norm = (X_train_cir - X_train_cir.mean(axis=1, keepdims=True)) / \
                   (X_train_cir.std(axis=1, keepdims=True) + 1e-8)
X_test_cir_norm = (X_test_cir - X_test_cir.mean(axis=1, keepdims=True)) / \
                  (X_test_cir.std(axis=1, keepdims=True) + 1e-8)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_cir_norm)
X_test_tensor = torch.FloatTensor(X_test_cir_norm)
y_train_tensor = torch.FloatTensor(y_train.values).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test.values).unsqueeze(1)

# Create datasets and loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

BATCH_SIZE = 256  # Large batch size for 5070 Ti
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

print(f"\n✓ Created DataLoaders")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")


In [ ]:
# SECTION 12: TRAIN TRANSFORMER


In [ ]:
print("\n" + "=" * 70)
print("TRAINING TRANSFORMER MODEL")
print("=" * 70)

# Training configuration
EPOCHS = 50
LEARNING_RATE = 1e-4

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': []
}

best_val_loss = float('inf')

print(f"\nTraining configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Device: {device}")
print(f"  Optimizer: AdamW")
print(f"  Scheduler: ReduceLROnPlateau")

# Training loop
for epoch in range(EPOCHS):
    # Training phase
    model.train()
    train_loss = 0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for batch_X, batch_y in train_pbar:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Validation phase
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        val_pbar = tqdm(test_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]')
        for batch_X, batch_y in val_pbar:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
            predictions = (outputs > 0.5).float()
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
            
            val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Calculate metrics
    train_loss /= len(train_loader)
    val_loss /= len(test_loader)
    
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f'\nEpoch {epoch+1}/{EPOCHS}:')
    print(f'  Train Loss: {train_loss:.4f}')
    print(f'  Val Loss:   {val_loss:.4f}')
    print(f'  Val Acc:    {val_acc:.4f}')
    print(f'  Val F1:     {val_f1:.4f}')
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODELS_DIR / 'transformer_best.pth')
        print(f'  ✓ Saved best model (val_loss={val_loss:.4f})')
    
    print('-' * 70)

print("\n✓ Training complete!")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Metrics
axes[1].plot(history['val_acc'], label='Accuracy', linewidth=2)
axes[1].plot(history['val_f1'], label='F1-Score', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_transformer_training.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '04_transformer_training.png'}")
plt.show()


In [ ]:
# SECTION 13: EVALUATE TRANSFORMER


In [ ]:
print("\n" + "=" * 70)
print("EVALUATING TRANSFORMER MODEL")
print("=" * 70)

# Load best model
model.load_state_dict(torch.load(MODELS_DIR / 'transformer_best.pth'))
model.eval()

# Get predictions
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in tqdm(test_loader, desc="Evaluating"):
        batch_X = batch_X.to(device)
        
        outputs = model(batch_X)
        predictions = (outputs > 0.5).float()
        
        all_preds.extend(predictions.cpu().numpy())
        all_probs.extend(outputs.cpu().numpy())
        all_labels.extend(batch_y.numpy())

all_preds = np.array(all_preds).flatten()
all_probs = np.array(all_probs).flatten()
all_labels = np.array(all_labels).flatten()

# Evaluation metrics
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['LOS', 'NLOS']))

print("\nConfusion Matrix:")
cm_transformer = confusion_matrix(all_labels, all_preds)
print(cm_transformer)

print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")
print(f"  Precision: {precision_score(all_labels, all_preds):.4f}")
print(f"  Recall:    {recall_score(all_labels, all_preds):.4f}")
print(f"  F1-Score:  {f1_score(all_labels, all_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(all_labels, all_probs):.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm_transformer, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['LOS', 'NLOS'], yticklabels=['LOS', 'NLOS'])
axes[0].set_title('Transformer Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr_trans, tpr_trans, _ = roc_curve(all_labels, all_probs)
roc_auc_trans = auc(fpr_trans, tpr_trans)
axes[1].plot(fpr_trans, tpr_trans, color='green', lw=2,
             label=f'Transformer (AUC = {roc_auc_trans:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Transformer')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_transformer_results.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '05_transformer_results.png'}")
plt.show()


In [ ]:
# SECTION 14: MODEL COMPARISON


In [ ]:
print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

# Compare models
comparison_data = {
    'Model': ['XGBoost', 'Transformer'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(all_labels, all_preds)
    ],
    'Precision': [
        precision_score(y_test, y_pred_xgb),
        precision_score(all_labels, all_preds)
    ],
    'Recall': [
        recall_score(y_test, y_pred_xgb),
        recall_score(all_labels, all_preds)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_xgb),
        f1_score(all_labels, all_preds)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_pred_proba_xgb),
        roc_auc_score(all_labels, all_probs)
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\nModel Performance Comparison:")
print(df_comparison.to_string(index=False))

# Save comparison
df_comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, df_comparison.iloc[0, 1:].values, width,
           label='XGBoost', color='orange', alpha=0.8)
axes[0].bar(x + width/2, df_comparison.iloc[1, 1:].values, width,
           label='Transformer', color='green', alpha=0.8)
axes[0].set_xlabel('Metrics')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=45)
axes[0].legend()
axes[0].set_ylim([0.85, 1.0])
axes[0].grid(alpha=0.3, axis='y')

# ROC curves comparison
axes[1].plot(fpr, tpr, color='orange', lw=2, 
            label=f'XGBoost (AUC = {roc_auc:.3f})')
axes[1].plot(fpr_trans, tpr_trans, color='green', lw=2,
            label=f'Transformer (AUC = {roc_auc_trans:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves Comparison')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_model_comparison.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '06_model_comparison.png'}")
plt.show()


In [ ]:
# SECTION 15: ATTENTION VISUALIZATION (BONUS)


In [ ]:
print("\n" + "=" * 70)
print("ATTENTION VISUALIZATION")
print("=" * 70)

def visualize_attention_gradients(model, sample_cir, label, device='cuda'):
    """Visualize what the transformer focuses on using gradients"""
    model.eval()
    
    # Prepare sample
    sample = torch.FloatTensor(sample_cir).unsqueeze(0).to(device)
    sample.requires_grad = True
    
    # Forward pass
    output = model(sample)
    
    # Backward to get gradients
    output.backward()
    
    # Attention is proportional to gradient magnitude
    attention = sample.grad.abs().squeeze().cpu().numpy()
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))
    
    # Original CIR
    axes[0].plot(sample_cir, linewidth=1, color='blue', alpha=0.7)
    axes[0].set_title(f'Original CIR - True Label: {"LOS" if label == 0 else "NLOS"}')
    axes[0].set_xlabel('Time (ns)')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(alpha=0.3)
    axes[0].axvline(np.argmax(sample_cir), color='red', linestyle='--', 
                    alpha=0.5, label='Peak')
    axes[0].legend()
    
    # Attention heatmap
    axes[1].plot(attention, color='red', linewidth=1.5, label='Attention Weight')
    axes[1].fill_between(range(len(attention)), attention, alpha=0.3, color='red')
    axes[1].set_title('Transformer Attention (What the model focuses on)')
    axes[1].set_xlabel('Time (ns)')
    axes[1].set_ylabel('Attention Weight')
    axes[1].grid(alpha=0.3)
    axes[1].axvline(np.argmax(attention), color='darkred', linestyle='--',
                    alpha=0.7, label='Max Attention')
    axes[1].legend()
    
    plt.tight_layout()
    return fig

# Visualize LOS and NLOS samples
print("Generating attention visualizations...")

# LOS sample
los_idx = np.where(y_test.values == 0)[0][0]
los_sample = X_test_cir_norm[los_idx]
los_label = y_test.values[los_idx]

fig_los = visualize_attention_gradients(model, los_sample, los_label, device)
plt.savefig(FIGURES_DIR / '07_attention_los.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '07_attention_los.png'}")
plt.show()

# NLOS sample
nlos_idx = np.where(y_test.values == 1)[0][0]
nlos_sample = X_test_cir_norm[nlos_idx]
nlos_label = y_test.values[nlos_idx]

fig_nlos = visualize_attention_gradients(model, nlos_sample, nlos_label, device)
plt.savefig(FIGURES_DIR / '07_attention_nlos.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {FIGURES_DIR / '07_attention_nlos.png'}")
plt.show()


In [ ]:
# SECTION 16: FINAL SUMMARY


In [ ]:
print("\n" + "=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)

summary = f"""
Dataset:
  Total Samples: {len(df_raw):,}
  LOS Samples: {(df_raw['Class'] == 0).sum():,}
  NLOS Samples: {(df_raw['Class'] == 1).sum():,}
  Features Extracted: {len(df_cir_features.columns)}
  
Training Configuration:
  Train Samples: {len(X_train):,}
  Test Samples: {len(X_test):,}
  Test Split: 20%
  
XGBoost Performance:
  Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}
  Precision: {precision_score(y_test, y_pred_xgb):.4f}
  Recall:    {recall_score(y_test, y_pred_xgb):.4f}
  F1-Score:  {f1_score(y_test, y_pred_xgb):.4f}
  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_xgb):.4f}

Transformer Performance:
  Parameters: {total_params:,}
  Accuracy:  {accuracy_score(all_labels, all_preds):.4f}
  Precision: {precision_score(all_labels, all_preds):.4f}
  Recall:    {recall_score(all_labels, all_preds):.4f}
  F1-Score:  {f1_score(all_labels, all_preds):.4f}
  ROC-AUC:   {roc_auc_score(all_labels, all_probs):.4f}

Best Model: {"Transformer" if accuracy_score(all_labels, all_preds) > accuracy_score(y_test, y_pred_xgb) else "XGBoost"}

Files Generated:
  - {len(list(FIGURES_DIR.glob('*.png')))} visualization files
  - {len(list(MODELS_DIR.glob('*')))} model files
  - {len(list(RESULTS_DIR.glob('*.csv')))} result CSV files
  - {len(list(RESULTS_DIR.glob('*.npy')))} numpy data files

All results saved to: {RESULTS_DIR.absolute()}
"""

print(summary)

# Save summary
with open(RESULTS_DIR / 'project_summary.txt', 'w') as f:
    f.write(summary)

print("\n" + "=" * 70)
print("✓ NOTEBOOK EXECUTION COMPLETE!")
print("=" * 70)
print(f"\nNext steps:")
print(f"1. Review results in: {RESULTS_DIR}")
print(f"2. Check visualizations in: {FIGURES_DIR}")
print(f"3. Use best model for your report")
print(f"4. Create presentation with attention visualizations!")
print("\nGood luck with your project! 🚀")
